# Davinci Notebook 12 — SFT Training, Chat Formatting & Tool Traces

Canonical chat serialization, assistant-only target masking, host-compatible tool-call message structure, training/validation splitting, and a fine-tuning loop. The exact token spellings are imported from the repository SFT module and must match the frozen SentencePiece tokenizer. Tool observations are context, not assistant targets.


In [ ]:
import json, torch, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset,DataLoader
random.seed(42)


## 1. Canonical message structure and serialization
Use a consistent serialization strategy for training and inference. The assistant tool-call payload follows the strict JSON protocol in `src/davinci/tool_call.py`: exactly `name` and `arguments`. The host executes only registered tools; tool observations use the `tool` role and are not assistant-authored targets.


In [ ]:
import sys, subprocess, base64
REPO_DIR=Path("/content/Davinci")
if not (REPO_DIR/".git").exists():
    from google.colab import userdata
    token=userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GitHub Secret GITHUB_TOKEN with Contents: read permission.")
    encoded=base64.b64encode(("x-access-token:"+token).encode()).decode()
    subprocess.run(["git","-c",f"http.extraheader=AUTHORIZATION: basic {encoded}","clone","--depth","1","https://github.com/keyhmoham/Davinci.git",str(REPO_DIR)],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci.sft import (
    SPECIAL_TOKENS, build_masked_example, causal_sft_loss, collate_masked,
    serialize_messages, split_examples, tool_call_content, tool_result_content,
)
REPO_COMMIT=subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip()
print("Imported SFT helpers from",REPO_COMMIT)

In [ ]:
SPECIAL=SPECIAL_TOKENS
example_trace=[
 {"role":"user","content":"What is 19 + 23?"},
 {"role":"assistant","content":tool_call_content("add",{"left":19,"right":23})},
 {"role":"tool","content":tool_result_content("add",{"sum":42})},
 {"role":"assistant","content":"42"},
]
serialized_example=serialize_messages(example_trace)
assert SPECIAL["tool_call"] in serialized_example and SPECIAL["tool_result"] in serialized_example
assert SPECIAL["end"] in serialized_example
print(serialized_example)

## 2. Assistant-only masking dataset
The caller must supply a tokenizer that can tokenize the serialized messages and identify the token boundaries. This template masks non-assistant tokens with `-100`; adapt it to the actual tokenizer's special token IDs and turn boundaries.

In [ ]:
messages=[
 {"role":"user","content":"سلام"},
 {"role":"assistant","content":"درود"},
]
# Token spellings are imported from the same repository module used by the model runtime.
example=build_masked_example(messages,tokenizer)
print("tokens:",len(example["input_ids"]),"assistant targets:",sum(x!=-100 for x in example["labels"]))

## 3. Dataset split and collator
Split by example ID or source document before training; avoid overlapping related examples across splits. This basic collator pads and uses labels=-100 for padding.

In [ ]:
train_examples,val_examples=split_examples(examples,val_fraction=0.05,seed=42)
print("train:",len(train_examples),"validation:",len(val_examples))
batch=collate_masked([build_masked_example(x["messages"],tokenizer) for x in train_examples[:4]],pad_id=tokenizer.pad_id())

## 4. SFT loop (generic model interface)
Assumes model(input_ids) returns logits `[B,T,V]`. This is a baseline loop; it omits packed-example masking, multi-GPU, and production scheduler/checkpoint policies.

In [ ]:
def sft_train_step(model,batch,optimizer,device,clip=1.0):
 ids,labels=[x.to(device) for x in batch]
 model.train()
 optimizer.zero_grad(set_to_none=True)
 output=model(ids,labels=labels)
 loss=output["loss"] if isinstance(output,dict) else causal_sft_loss(output,labels)
 if loss is None: raise ValueError("Batch contains no assistant next-token targets")
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
 optimizer.step()
 return float(loss.detach())

print("SFT step is compatible with DavinciForCausalLM(input_ids, labels=labels).")

## 5. Dataset audit before fine-tuning
- Check source licenses and redistribution conditions.
- Deduplicate against pretraining and held-out data.
- Review generated reasoning for correctness and leakage.
- Remove personally identifying or confidential content.
- Keep tool traces structured and separate observations from model-authored messages.
- Evaluate separately on Persian, English, bilingual and tool-use tasks.


## 6. Tool-trace protocol alignment
Before training, parse every assistant tool-call JSON body with the same strict protocol as the inference runtime, validate tool names/arguments against the tool manifest, and check that each tool result corresponds to a preceding call. If `<|tool_call|>` is a dedicated token, detect it as a structural boundary and pass only the following JSON body to the parser. Do not train the model to emit host-only secrets, stack traces, or unredacted tool output. Tool-call assistant messages are supervised targets; tool observations remain context only in the assistant-only masking template above.
